In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "storageproject99")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/programa/ps_prog_tbl.csv"

In [0]:
programa_schema = StructType(fields=[StructField("INSTITUTION", StringType(), True),
                                     StructField("EFFDT", StringType(), True),
                                     StructField("EFF_STATUS", StringType(), True),
                                     StructField("ACAD_PROG", StringType(), True),
                                     StructField("DESCR", StringType(), True),
                                     StructField("DESCRSHORT", StringType(), True),
                                     StructField("ACAD_CAREER", StringType(), True),
                                     StructField("ACAD_GROUP", StringType(), True)
                                     ])

In [0]:
df_programa_schema_read = spark.read.option('header', True)\
                                    .schema(programa_schema)\
                                    .csv(ruta)

In [0]:
df_programa_ingestion_date = df_programa_schema_read.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_programa_final = df_programa_ingestion_date.select("INSTITUTION",
                                                      "EFFDT",
                                                      "EFF_STATUS",
                                                      "ACAD_CAREER",
                                                      "ACAD_PROG",
                                                      "DESCR",
                                                      "DESCRSHORT",
                                                      "ACAD_GROUP",
                                                      "INGESTION_DATE"
                                                      )

In [0]:
df_programa_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.programa")